# 第16讲：让已有 Skill 随新任务迭代

**迭代**：从一次完成的调查里判断哪些做法值得写回同一个方法，再用新现场与旧任务决定继续修订还是保持原版。

先完成下面的保存轨迹练习，不需要模型或 Hermes；后半段保留本次原生运行入口。保存轨迹来自本讲已有 east、west、single 构造现场，不是生产事故，也不代表你的修改已运行。

In [ ]:
from pathlib import Path
import importlib.util
import os
from IPython.display import Markdown, display

CASE_HOME = Path.cwd().resolve()
if not (CASE_HOME / "iteration_case.py").is_file():
    CASE_HOME = CASE_HOME / "examples/16-skill-incremental-patch"
spec = importlib.util.spec_from_file_location("iteration_case", CASE_HOME / "iteration_case.py")
caseview = importlib.util.module_from_spec(spec)
spec.loader.exec_module(caseview)
display(Markdown(caseview.render("origin")))


## 先决定：这次成功值得怎样写回方法？

east 已经查清；先对照工具回执和原版步骤，不把“原来没写关联分支”直接当成任务失败。哪些判断属于可复用方法，哪些只是这次现场的结论？

在下一格写出最小修改、适用条件、必须保留未知的边界，以及不能损坏的旧路径；也可以决定保持原版并说明依据。写好后再打开后面的候选。

In [ ]:
proposal = {
    "修改或保持原版": "",
    "轨迹依据": "",
    "最小方法改动": "",
    "适用与停止判断的条件": "",
    "要保留的旧路径": "",
}
# 在上面写下你的判断；程序不替你评分或采用 Skill。


## 再看：原生候选到了新现场会怎样？

写好初稿后，将下一格的 `SHOW_CANDIDATE` 改为 True。查看同名方法差异、原生读写调用和两个空历史新会话的回执与原回答。

对照你提出的修改：哪些新增步骤值得保留？哪些规则需要继续改？什么结果会让你保持原版？把决定指向方法原句和调用编号，再写出下一次回放要观察的行为。结构检查通过仍需阅读实际内容。

In [ ]:
SHOW_CANDIDATE = False
if SHOW_CANDIDATE:
    display(Markdown(caseview.render("candidate")))
else:
    print("候选与后续任务尚未展开；先写下你自己的方法修改。")


In [ ]:
revision_decision = {
    "保留的步骤及依据": "",
    "继续修订或保持原版": "",
    "方法原句与回执编号": "",
    "下一次要观察的行为": "",
}


## 最后看：收紧一条规则以后，还需要改什么？

完成上一格判断后，将 `SHOW_REVISION` 改为 True，查看已有的审阅代理补丁及它的新会话结果。决定这条修改是否暂时保留，并说明还要不要继续改方法。

若方法本身把线索当成结论，修订规则；若轨迹未执行已有步骤，先查执行；若报告误引回执，先纠正引用。三者都可能存在，不能把每次报告错误直接追加成长期规则。不要只凭补丁哈希或 `passed` 作决定。

In [ ]:
SHOW_REVISION = False
if SHOW_REVISION and SHOW_CANDIDATE:
    display(Markdown(caseview.render("revision")))
elif SHOW_REVISION:
    print("先展开原生候选与回放，再查看后一次修订。")
else:
    print("后一次修订尚未展开；先决定原候选哪些内容值得保留。")


In [ ]:
final_decision = {
    "这条修订是否暂时保留": "",
    "迁移与旧路径的依据": "",
    "还需改方法还是处理执行或引用": "",
    "仍然不能下的结论": "",
}


## 将判断带到你自己的运行

保存的两次回放只覆盖各自使用的版本；你的纸面补丁若不同，不能借它们声称已通过。也没有原版在同一 west 现场的配对，不能据此推断全面增益。

下面保留实时调查、原生后台修订与新会话回放。先按 [README](README.md#运行与停止) 安装依赖，再将 `RUN_NATIVE_WORKSHOP` 改为 True；只有进一步打开 `RUN_MODEL` 才调用模型并产生费用。这里产生新运行，与前面的保存轨迹分开。

In [ ]:
RUN_NATIVE_WORKSHOP = (os.environ.get("LESSON16_NATIVE_WORKSHOP") == "1"
                       or os.environ.get("LESSON16_RUN_MODEL") == "1")
# 手动进入本次原生运行时可改为 True；默认只做保存轨迹练习。


In [ ]:
if RUN_NATIVE_WORKSHOP:
    from pathlib import Path
    import ast
    import json
    import os
    import signal
    import subprocess
    from IPython.display import Code, display

    if "process" in globals() and process is not None:
        stop_owned()
    HERE = Path.cwd().resolve()
    if not (HERE / "native_workshop.py").exists():
        HERE = HERE / "examples/16-skill-incremental-patch"
    assert (HERE / "native_workshop.py").is_file()
    REPO = HERE.parents[1]
    HERMES_SRC = Path(os.environ.get("HERMES_SRC", REPO / ".deps/hermes-agent")).resolve()
    HERMES_PYTHON = Path(os.environ.get("HERMES_PYTHON", HERMES_SRC / ".venv/bin/python"))
    assert HERMES_PYTHON.is_file(), "请先运行仓库 scripts/setup_deps.sh"
    env = dict(os.environ, HERMES_SRC=str(HERMES_SRC), PYTHONDONTWRITEBYTECODE="1",
               PYTHONPATH=str(HERMES_SRC) + os.pathsep + os.environ.get("PYTHONPATH", ""))
    process = None
    live_result = None

    def stop_owned():
        global process
        if process is not None and process.poll() is None:
            os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait(timeout=10)
        captured = ""
        if process is not None and process.stdout is not None:
            try:
                captured, _ = process.communicate(timeout=1)
            except (ValueError, subprocess.TimeoutExpired):
                pass
            process.stdout.close()
        process = None
        return captured

else:
    print("本次原生运行未启用；保存轨迹不算本次执行结果。")


## 本次原生运行的起点

下面读取本讲给定的方法与观察目标，不请求模型。原生执行函数可通过 `SHOW_IMPLEMENTATION` 展开；前台使用空历史，后台通过 `spawn_background_review_thread()` 执行复盘。


In [ ]:
if RUN_NATIVE_WORKSHOP:
    source = (HERE / "native_workshop.py").read_text()
    tree = ast.parse(source)
    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name in ("foreground", "review_job"):
            display(Code(ast.get_source_segment(source, node), language="python"))

else:
    print("本次原生运行未启用；保存轨迹不算本次执行结果。")


## 阅读旧方法与构造现场

旧 `oncall-alert-triage` 只覆盖单告警。事件 east 中两项告警共用故障组件，另一项同期告警有独立原因。工具返回当前指标、日志、依赖、调用链和探测结果，不能仅凭时间接近合并。

下面不请求模型。自检回执全部是明确构造的校验器输入，不依赖历史回放。


In [ ]:
if RUN_NATIVE_WORKSHOP:
    import scenario
    print("课程给定的旧方法：\n", scenario.SEED)
    print("工具说明：\n", scenario.REFERENCE)
    for script in ("native_workshop.py", "selfcheck.py"):
        checked = subprocess.run([str(HERMES_PYTHON), "-B", str(HERE / script)],
                                 cwd=HERE, env=env, text=True, capture_output=True)
        print(checked.stdout)
        print(checked.stderr)
        checked.check_returncode()

else:
    print("本次原生运行未启用；保存轨迹不算本次执行结果。")


## 新故障调查、复盘修订与新旧回放

设置当前进程的 `GLM_API_KEY`，把 `RUN_MODEL` 改为 `True`。默认 `glm-5.2`、官方接口，可用环境变量覆盖。调用会产生 API 费用。

前台先读旧 Skill 再调查 east。复盘依照原生提示和 `scenario.FOCUS` 先读后改，优先 `skill_manage(action="patch")` 修订同名主文件；引用按需修改并回读。脚本不提供新方法正文。

本格默认在独立教学库自动采用完整候选，记录 `teaching-auto`，不算人工审批。如需逐份确认，请在终端使用 `bash run.sh --live --no-auto-approve`，输入候选完整哈希；本格不会替你作人工确认。

随后创建两个空历史 Session：west 检验关联调查并保留未知，single 检验旧单告警路径。若无可靠修改，保留 `no-update` 与原生说明，不强行制造补丁。


In [ ]:
if RUN_NATIVE_WORKSHOP:
    from uuid import uuid4
    import sys
    sys.path.insert(0, str(HERE.parent))
    from notebook_native_results import collect_run, save_observation
    from common import scrub
    RUN_MODEL = os.environ.get("LESSON16_RUN_MODEL") == "1"  # 交互运行时可改为 True
    live_result = None
    if not RUN_MODEL:
        print("尚未运行模型；上面的现场检查不代表 Skill 学习成功。")
    else:
        for name in ("GLM_API_KEY", "GLM_MODEL", "GLM_BASE_URL"):
            if name in os.environ:
                env[name] = os.environ[name]
        current_run = Path(os.environ.get("LESSON16_RUN_DIR") or (HERE / "runtime" / ("native-notebook-" + uuid4().hex)))
        if current_run.exists() or current_run.is_symlink():
            raise ValueError("本次运行目录已存在，请使用新的运行目录，不读取历史记录。")
        validation_id = os.environ.get("LESSON16_VALIDATION_ID")
        child, output, process_error = None, "", None
        try:
            process = child = subprocess.Popen(
                [str(HERMES_PYTHON), "-B", str(HERE / "native_workshop.py"), "--live",
                 "--run-dir", str(current_run)],
                cwd=HERE, env=env, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT, text=True, start_new_session=True)
            output, _ = child.communicate(timeout=900)
        except (Exception, KeyboardInterrupt) as error:
            process_error = type(error).__name__ + ": " + str(error)
            partial = getattr(error, "output", "") or ""
            output = partial.decode(errors="replace") if isinstance(partial, bytes) else partial
        finally:
            captured = stop_owned()
            if captured:
                output = captured
        if current_run.is_dir():
            (current_run / "subprocess.log").write_text(scrub(output))
        print(scrub(output))
        live_result = collect_run(current_run, validation_id,
                                  child.returncode if child else None, process_error)
        save_observation(HERE, live_result)
        print("本次状态：", live_result["status"], "；继续查看本轮已有记录。")

else:
    print("本次原生运行未启用；保存轨迹不算本次执行结果。")


## 核对补丁、采用和新旧能力

下面只读取为刚才进程分配的独立目录。非零退出或超时时仍展示本轮已有记录和错误；尚未进入的复盘、采用或回放阶段明确显示未取得，不能当成学习成功。逐条核对原生调用、成功回执、文件差异和最终回读，确认主文件保留单告警步骤，把多告警条件融入流程，而不是只追加教训。

两份未来任务记录应显示新 Session、空启动历史，以及查询前读取了采用版本的完整正文。程序检查调用和构造业务结果；是否形成通用方法、范围是否合适，仍需你阅读内容判断。


In [ ]:
if RUN_NATIVE_WORKSHOP:
    from notebook_native_results import show_observation
    if live_result is None:
        print("尚无本次模型运行，不展示历史 Skill 充当本次结果。")
    else:
        show_observation(live_result)

else:
    print("本次原生运行未启用；保存轨迹不算本次执行结果。")


## 收尾与继续思考

收尾只停止本格启动的进程并保留本次记录。源 Notebook 没有预置执行输出；前面的保存轨迹不用于本次实时运行验收。

跨实例任务分布与修订协调由外层 Harness 负责。两个实例基于不同版本提出补丁时，怎样排队、检测冲突并重新审阅？两个回放以后，还要保留哪些旧任务来检查退步？


In [ ]:
if RUN_NATIVE_WORKSHOP:
    stop_owned()
